# First look at a SANS dataset: model-free analysis

`iquana.modelfree` answers the first questions about a curve before any model
is chosen:

- **Porod**: the Porod constant and the flat background at high Q
- **Guinier**: Rg and I(0), with an automatic Q window and a check of the q·Rg limit
- **Kratky**: is the particle compact?
- **Invariant**: Q*, the volume fraction and the specific surface
- **Power law**: the slope over a chosen window

The results are plain numbers, ready as starting values for a model fit.
Part 1 uses simulated data, where the answers are known; part 2 uses a
measured protein dataset.

## 1. Simulated polydisperse spheres

Spheres of number-mean radius 50 Å with a 20 % Gaussian width, at volume
fraction φ = 1 % and SLD difference 3 × 10⁻⁶ Å⁻², on absolute scale (cm⁻¹).

In [ ]:
import math

from iquana import IQuana, examples, modelfree

radius, width, phi, delta_sld = 50.0, 0.2, 0.01, 3.0
data = examples.simulate(
    'sphere', radius=radius, radius_pd=width, scale=phi, sld=4.0, sld_solvent=1.0,
    background=0.001, qmin=0.002, qmax=0.6, npoints=300, noise=0.01, seed=1,
)

### Porod first

Fitting I = Kp·q⁻⁴ + B at high Q gives the Porod constant and the background
in one step. Every later analysis should subtract this same background:
`kratky()` and `invariant()` refuse results computed with different ones.
A free power law over the same window checks that the exponent really is 4.

In [ ]:
p = modelfree.porod(data, qmin=0.15)
print(p.format_summary())
print(f'Simulated background: {data.truth["background"]}')

In [ ]:
p.plot(data)

### Guinier

With no `qmax`, the window grows from the lowest point and stops at the
largest window with q_max·Rg ≤ 1.3. The low-Q edge is never trimmed
automatically: cutting an upturn is a decision for `qmin`.

The report has the same form as a model fit's. Its notes give q·Rg at both
edges and the curvature of ln I against q², which separates the bending a
real particle has from a downturn (repulsion) or an upturn (elongation,
polydispersity, aggregation).

In [ ]:
g = modelfree.guinier(data, background=p.background)
g.report()

In [ ]:
g.plot(data)

Rg of a polydisperse sample is intensity-weighted, so `sphere_radius`
(√(5/3)·Rg) is larger than the number-mean radius of 50 Å. It is still a
good starting value.

In [ ]:
print(f'Rg = {g.rg:.2f} ± {g.rg_err:.2f} Å, sphere radius {g.sphere_radius:.1f} Å')

### Kratky

The dimensionless form (q·Rg)²·I/I(0) against q·Rg marks the Guinier-law peak
at (√3, 3/e). A homogeneous sphere peaks a little below and to the left of it,
at (1.61, 1.03); a flexible chain rises to a plateau instead.

In [ ]:
k = modelfree.kratky(data, guinier=g)
print(f'Peak at q·Rg = {k.peak_position:.3f}, height {k.peak_height:.3f}')
k.plot()

### Invariant

Q* = ∫q²(I − B)dq: the measured part by trapezoid, the Guinier result below
the first point and the Porod result above the last. `delta_sld` (in
10⁻⁶ Å⁻², like every SLD in IQuana) gives the volume fraction and the
Porod-route specific surface; `volume_fraction` gives the scale-free specific
surface and a check of the absolute scale.

The error of Q* is propagated once from the intensities, through the
trapezoid and through both fits, because they share data points.

In [ ]:
inv = modelfree.invariant(data, low_q=g, high_q=p, delta_sld=delta_sld, volume_fraction=phi)
print(inv.format_summary())
print(f'\nExpected Q* = {2 * math.pi**2 * delta_sld**2 * phi * 1e-4:.4g}')

In [ ]:
inv.plot()

### Power law over a chosen window

`fit_background=True` fits A·q⁻ⁿ + B in linear space; the default fixed
background fits a line in log-log space, which is biased near the background
level (the `low_signal` warning says when).

In [ ]:
pl = modelfree.power_law(data, qmin=0.15, qmax=0.6, fit_background=True)
print(pl.format_summary())

### Starting values for a model fit

In [ ]:
fitter = IQuana()
fitter.set_data(data)
fitter.set_model('sphere')
fitter.set_param('radius', value=g.sphere_radius, min=1, max=500)
fitter.set_param('background', value=p.background, min=0, max=1)

## 2. Measured data: apoferritin

A real protein dataset with a noisy high-Q tail. The checks matter here: read
the warnings before using the numbers.

In [ ]:
protein = examples.load('protein')
p2 = modelfree.porod(protein, qmin=0.15)
g2 = modelfree.guinier(protein, background=p2.background)
print(g2.format_summary())

In [ ]:
inv2 = modelfree.invariant(protein, low_q=g2, high_q=p2)
print(inv2.format_summary())

Two warnings, both worth acting on:

- **`curvature_positive`**: ln I bends upward in the Guinier window. For
  apoferritin, a compact shell, that points to some aggregation; the Rg is
  larger than the shell alone would give. Try `qr_limit=1.0`, or move `qmin`.
- **`extrapolation_high`**: a fifth of Q* comes from the Porod tail, because
  the high-Q data are noisy and end early. Q* is only as good as that tail.

## Things to know

- **One background.** Fit Porod first and pass `background=p.background` to
  `guinier()`; chained calls with different backgrounds raise.
- **Slit-smeared data** (USANS) is refused by every function; pinhole dQ is
  ignored, with a `dq_ignored` warning.
- **Warnings** are stored on each result as `(code, message)` pairs
  (`result.warning_codes`) and also emitted through Python's `warnings`.
- **Units**: Q in 1/Å, `delta_sld` in 10⁻⁶ Å⁻², absolute quantities need cm⁻¹.

See `docs/usage.md` ("Model-free analysis") and
`examples/modelfree_example.py`.